## Generating summary & bias for each article:

In [ ]:
!git clone https://github.com/probcomp/hfppl.git
!cd hfppl && pip install . && cd ..

In [2]:
import sys
sys.path.append("/data/cb/scratch/bfefferm/NLP-Project/hfppl")

In [4]:
import pandas as pd
import csv
import os
from hfppl import Model, LMContext, TokenCategorical, CachedCausalLM, smc_steer, smc_standard
from score import compute_pbf_score, compute_pbi_score
from smc_steer_summary import bias_model_factory, TwistModel, gen_summary

**Loading Dataset:**

In [5]:
dataset = pd.read_csv('../POLITICS_finetuning/processed_data.csv')

In [6]:
dataset

,title,body,stance
0,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
1,Mary Cheney: Sister Is 'Dead Wrong' On Gay Mar...,"Mary Cheney, the younger sister of Wyoming U.S...",center
2,IRS official who refused to testify facing mor...,The IRS official who refused to testify at a H...,center
3,White House Plays Down Data Program,WASHINGTON — The Obama administration tried Sa...,center
4,N.R.A. Details Plan for Armed School Guards,Report Sees Guns as Path to Safety in Schools\...,center
...,...,...,...
295,Nancy Pelosi Re-Elected House Minority Leader,WASHINGTON ― House Minority Leader Nancy Pelos...,right
296,Nancy Pelosi Beats Back House Democratic Leade...,WASHINGTON — House Democrats on Wednesday reje...,center
297,Obama Will Meet With Sanders On Thursday,WASHINGTON -- With presumptive Democratic pres...,left
298,"Clinton Is 'Sane' And 'Competent,' Unlike Trum...",PHILADELPHIA ― Americans should vote for Hilla...,center


In [ ]:
# need to login to use llama
from huggingface_hub import notebook_login

notebook_login()

**Specifying file paths:**

In [18]:
# Model paths:
import torch

print('Loading bias model...')
bias_model_path = # TODO: Fill in
bias_tokenizer_path = # TODO: Fill in

bias_model = bias_model_factory(bias_model_path, bias_tokenizer_path)

print('Loaded!')


# Specifying model name:
# llm_model_name = 'gpt2'

# from transformers import GPT2Tokenizer, GPT2LMHeadModel

# gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
# gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2',  load_in_8bit=True, device_map='auto')

from transformers import LlamaForCausalLM, LlamaTokenizer

llama_tokenizer = LlamaTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf", torch_dtype=torch.float32)
llama_model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf", torch_dtype=torch.float32, device_map='auto')
llama_tokenizer.model_max_length = 1024

llm = CachedCausalLM(llama_model, llama_tokenizer)


# Loading llm:
# llm = CachedCausalLM.from_pretrained(llm_model_name, load_in_8bit=True)

You are loading your model in 8bit or 4bit but no linear modules were found in your model. this can happen for some architectures such as gpt2 that uses Conv1D instead of Linear layers. Please double check your model architecture, or submit an issue on github if you think this is a bug.


**Iterating over each summary, predicting bias, & saving to `.csv`:**

In [19]:
num_summaries = 3

In [ ]:
# Open file  
with open('../POLITICS_finetuning/processed_data.csv') as file_obj: 
    # Create reader object by passing the file  
    # object to reader method 
    reader_obj = csv.reader(file_obj) 
    with open('../summaries/llama-smc.csv', 'w') as f:  
        # Initialize writer object:
        writer_obj = csv.writer(f)
        # The fields of this file are
        # ['title', 'body', 'stance']
        # Iterate over each row in the .csv,
        # skipping the first row (pertaining to field / column)
        next(reader_obj)
        writer_obj.writerow(['Title', 'Summary', 'Predicted Bias', 'Stance'])
        for row in reader_obj: 
            # Store title:
            title = row[0]
            # Store article:
            article = row[1]
            # Ensuring that articles fit within maximum length
            # For each stance:
            for stance in ['left', 'center', 'right']:
                for i in range(num_summaries):
                    summary = await gen_summary('llama', llm, bias_model, TwistModel, article, stance)
                    # For each summary, predict its bias:
                    # pred_bias, logits = bias_model(summary)
                    writer_obj.writerow([title, summary, None, stance])
                    
    writer_obj.close()